# Clase 1 · Medir la dispersión: varianza, desviación estándar y estandarización

**Módulo 1 · Describir datos**

### Objetivos de la clase
Al terminar podrás:
1. Calcular a mano la **varianza** y la **desviación estándar** de un conjunto pequeño de datos.
2. Explicar por qué se elevan al cuadrado las diferencias y por qué se divide entre $n - 1$.
3. Entender que una **proporción** es el promedio de una variable que sólo vale 0 o 1.
4. **Estandarizar** una variable (puntaje z), que es lo que hace `StandardScaler` en Machine Learning.

### Lo que ya sabes (de tu diagnóstico)
- Calculas bien la media y la mediana, y sabes que la mediana resiste mejor los valores extremos.
- Sabes que la desviación estándar mide dispersión. En esta clase se precisa un detalle: se mide **respecto a la media**, no respecto a la mediana.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

---
## Paso 1 · Los datos de ejemplo

Usamos las edades del diagnóstico. Son 5 datos para poder hacer cada cuenta a mano.

In [2]:
edades = pd.Series([20, 22, 24, 26, 80])
n = len(edades)
print("Edades:", edades.tolist())
print("Número de datos (n):", n)

Edades: [20, 22, 24, 26, 80]
Número de datos (n): 5


**Comprobación:** hay 5 datos y ninguno falta.

In [3]:
print("Datos faltantes:", edades.isna().sum())

Datos faltantes: 0


---
## Paso 2 · La media

La media es la suma de los datos dividida entre cuántos datos hay:

$$\bar x = \frac{20 + 22 + 24 + 26 + 80}{5} = \frac{172}{5} = 34.4$$

In [4]:
suma_edades = edades.sum()
media_edades = suma_edades / n
print("Suma de las edades:", suma_edades)
print("Media de las edades:", media_edades)

Suma de las edades: 172
Media de las edades: 34.4


---
## Paso 3 · ¿Qué tan lejos está cada dato de la media?

Para medir la dispersión, primero se calcula la **desviación** de cada dato: el dato menos la media.

| Edad | Edad − media |
|---|---|
| 20 | 20 − 34.4 = −14.4 |
| 22 | 22 − 34.4 = −12.4 |
| 24 | 24 − 34.4 = −10.4 |
| 26 | 26 − 34.4 = −8.4 |
| 80 | 80 − 34.4 = 45.6 |

Un valor negativo indica que el dato está por debajo de la media; uno positivo, que está por encima.

In [5]:
desviaciones = edades - media_edades
print("Desviación de cada edad respecto a la media:", desviaciones.round(2).tolist())

Desviación de cada edad respecto a la media: [-14.4, -12.4, -10.4, -8.4, 45.6]


### El problema: las desviaciones siempre suman cero

Si sumamos las desviaciones para obtener una "dispersión total", el resultado es 0. Esto pasa **siempre**, con cualquier conjunto de datos, porque la media es justo el punto donde las distancias negativas y positivas se compensan.

In [6]:
suma_desviaciones = desviaciones.sum()
print("Suma de las desviaciones:", round(suma_desviaciones, 10))

Suma de las desviaciones: 0.0


**Conclusión del paso:** sumar las desviaciones no sirve para medir dispersión. Hay que quitarles el signo. La forma que usa la estadística es **elevarlas al cuadrado**: un número al cuadrado siempre es positivo.

---
## Paso 4 · La varianza

Se elevan al cuadrado las desviaciones:

| Desviación | Desviación al cuadrado |
|---|---|
| −14.4 | 207.36 |
| −12.4 | 153.76 |
| −10.4 | 108.16 |
| −8.4 | 70.56 |
| 45.6 | 2079.36 |
| **Suma** | **2619.20** |

La **varianza** es esa suma dividida entre $n - 1$:

$$s^2 = \frac{\sum (x_i - \bar x)^2}{n - 1} = \frac{2619.2}{4} = 654.8$$

| Símbolo | Significado |
|---|---|
| $s^2$ | Varianza de la muestra |
| $x_i$ | Cada dato (la $i$ indica "el dato número $i$") |
| $\bar x$ | La media |
| $\sum$ | Sumar todos los términos |
| $n - 1$ | Número de datos menos uno |

In [7]:
desviaciones_al_cuadrado = desviaciones ** 2
suma_cuadrados = desviaciones_al_cuadrado.sum()
varianza_edades = suma_cuadrados / (n - 1)
print("Desviaciones al cuadrado:", desviaciones_al_cuadrado.round(2).tolist())
print("Suma de las desviaciones al cuadrado:", round(suma_cuadrados, 2))
print("Varianza (dividiendo entre n - 1):", round(varianza_edades, 2))

Desviaciones al cuadrado: [207.36, 153.76, 108.16, 70.56, 2079.36]
Suma de las desviaciones al cuadrado: 2619.2
Varianza (dividiendo entre n - 1): 654.8


### ¿Por qué se divide entre $n - 1$ y no entre $n$?

Las desviaciones se miden respecto a $\bar x$, que se calculó con **los mismos datos**. Por eso los datos de una muestra quedan, en promedio, un poco más cerca de $\bar x$ que de la media real de la población. Si se divide entre $n$, la varianza sale **un poco más pequeña** de lo que es en la población. Dividir entre $n - 1$ compensa ese efecto.

- Si tienes **toda la población**, se divide entre $n$.
- Si tienes **una muestra** (el caso normal), se divide entre $n - 1$.

Con muestras grandes la diferencia es mínima. Con 5 datos sí se nota:

In [8]:
varianza_entre_n = suma_cuadrados / n
print("Varianza dividiendo entre n:", round(varianza_entre_n, 2))
print("Varianza dividiendo entre n - 1:", round(varianza_edades, 2))

Varianza dividiendo entre n: 523.84
Varianza dividiendo entre n - 1: 654.8


### El problema de la varianza: sus unidades

Las edades están en **años**. Al elevar al cuadrado, la varianza queda en **años al cuadrado** (654.8 años²). Esa unidad no se puede interpretar directamente. Por eso existe el siguiente paso.

---
## Paso 5 · La desviación estándar

La **desviación estándar** es la raíz cuadrada de la varianza. Con la raíz, la unidad regresa a **años**:

$$s = \sqrt{s^2} = \sqrt{654.8} = 25.59 \text{ años}$$

**Interpretación:** en promedio, las edades se alejan unos 25.6 años de la media. Es un valor grande porque el 80 está muy lejos del resto.

In [9]:
desviacion_estandar_edades = np.sqrt(varianza_edades)
print("Desviación estándar:", round(desviacion_estandar_edades, 2), "años")

Desviación estándar: 25.59 años


**Comprobación:** pandas calcula lo mismo. Por defecto, `.var()` y `.std()` dividen entre $n - 1$.

In [10]:
print("Varianza con pandas:", round(edades.var(), 2))
print("Desviación estándar con pandas:", round(edades.std(), 2))

Varianza con pandas: 654.8
Desviación estándar con pandas: 25.59


### ¿Cuánto influye el 80?

Se repite el cálculo sin el 80:

In [11]:
edades_sin_80 = pd.Series([20, 22, 24, 26])
print("Media sin el 80:", edades_sin_80.mean())
print("Desviación estándar sin el 80:", round(edades_sin_80.std(), 2), "años")

Media sin el 80: 23.0
Desviación estándar sin el 80: 2.58 años


**Conclusión del paso:** al quitar un solo dato extremo, la desviación estándar baja de 25.6 a 2.6 años. Igual que la media, la desviación estándar es **muy sensible a los valores extremos**, porque las desviaciones grandes, al elevarse al cuadrado, crecen mucho (45.6² = 2079.36).

---
## Paso 6 · Una proporción es una media de ceros y unos

En tu diagnóstico no supiste qué es una proporción. Es la fracción del total que cumple una condición:

$$\text{proporción} = \frac{\text{casos que cumplen}}{\text{total}} = \frac{30}{100} = 0.30$$

0.30 y 30 % son el mismo número escrito de dos formas (un porcentaje es la proporción multiplicada por 100).

Ahora un detalle importante: si cada cliente vale **1 si compró y 0 si no compró**, la media de esa columna **es** la proporción. La suma de la columna cuenta los unos y al dividir entre $n$ se obtiene la fracción.

In [12]:
compras = pd.Series([1] * 30 + [0] * 70)
proporcion_compras = compras.mean()
print("Total de clientes:", len(compras))
print("Clientes que compraron:", compras.sum())
print("Proporción (media de la columna 0/1):", proporcion_compras)

Total de clientes: 100
Clientes que compraron: 30
Proporción (media de la columna 0/1): 0.3


Esto es lo que se hizo en el proyecto con la columna `Exited`: la proporción de abandono es la media de esa columna.

In [13]:
RAIZ = Path.cwd().parent if Path.cwd().name == "clases" else Path.cwd()
clientes = pd.read_csv(RAIZ / "data" / "raw" / "Churn_Modelling.csv")
print("Número de clientes:", len(clientes))
print("Clientes que abandonaron:", clientes["Exited"].sum())
print("Proporción de abandono:", clientes["Exited"].mean())

Número de clientes: 10000
Clientes que abandonaron: 2037
Proporción de abandono: 0.2037


**Comprobación:** la columna sólo tiene ceros y unos, así que su media es una proporción válida.

In [14]:
print("Valores distintos en Exited:", sorted(clientes["Exited"].unique().tolist()))

Valores distintos en Exited: [0, 1]


---
## Paso 7 · Conexión con Machine Learning: estandarizar variables

En tu curso de ML vas a usar `StandardScaler`. Lo que hace es convertir cada dato en su **puntaje z**:

$$z = \frac{x - \bar x}{s}$$

El puntaje z dice **a cuántas desviaciones estándar está un dato de la media**. Un $z = 2$ significa "2 desviaciones estándar por encima de la media"; un $z = -1$, "1 desviación estándar por debajo".

**¿Por qué se hace en ML?** Las variables tienen escalas muy distintas: la edad va de 18 a 92 y el saldo de 0 a 250,000. Muchos modelos (regresión logística, k vecinos más cercanos, redes neuronales) dan más peso a la variable con números más grandes sólo por su escala. Después de estandarizar, todas las variables tienen media 0 y desviación estándar 1, y quedan comparables.

Se estandariza la edad de los 10,000 clientes:

In [15]:
edad_clientes = clientes["Age"]
media_edad_clientes = edad_clientes.mean()
desviacion_edad_clientes = edad_clientes.std()
print("Media de la edad:", round(media_edad_clientes, 2))
print("Desviación estándar de la edad:", round(desviacion_edad_clientes, 2))

edad_estandarizada = (edad_clientes - media_edad_clientes) / desviacion_edad_clientes
print("Primeras 5 edades:", edad_clientes.head().tolist())
print("Primeros 5 puntajes z:", edad_estandarizada.head().round(2).tolist())

Media de la edad:

 38.92
Desviación estándar de la edad: 10.49
Primeras 5 edades: [42, 41, 42, 39, 43]
Primeros 5 puntajes z: [0.29, 0.2, 0.29, 0.01, 0.39]


**Comprobación:** después de estandarizar, la media debe ser 0 y la desviación estándar 1.

In [16]:
print("Media de la edad estandarizada:", round(edad_estandarizada.mean(), 6))
print("Desviación estándar de la edad estandarizada:", round(edad_estandarizada.std(), 6))

Media de la edad estandarizada: 0.0
Desviación estándar de la edad estandarizada: 1.0


Lectura del primer cliente: tiene 42 años y su puntaje z es 0.29. Está 0.29 desviaciones estándar por encima de la edad media (38.9 años).

---
## Resumen de la clase

| Concepto | Fórmula | Qué mide |
|---|---|---|
| Media | $\bar x = \dfrac{\sum x_i}{n}$ | El centro de los datos |
| Varianza | $s^2 = \dfrac{\sum (x_i - \bar x)^2}{n-1}$ | Dispersión, en unidades al cuadrado |
| Desviación estándar | $s = \sqrt{s^2}$ | Dispersión, en las unidades originales |
| Proporción | $\hat p = \dfrac{x}{n}$ = media de una columna 0/1 | Fracción del total que cumple una condición |
| Puntaje z | $z = \dfrac{x - \bar x}{s}$ | A cuántas desviaciones estándar está un dato de la media |

---
## Ejercicios

Resuélvelos **a mano** (con calculadora) y después compruébalos en una celda nueva.

**Ejercicio 1.** Calificaciones de un examen: 6, 7, 8, 9, 10.
a) Calcula la media.
b) Calcula la varianza (dividiendo entre $n - 1$).
c) Calcula la desviación estándar.

**Ejercicio 2.** Con los mismos datos del ejercicio 1, calcula el puntaje z de la calificación 10. Escribe en una frase qué significa ese número.

**Ejercicio 3.** De 8 clientes, la columna "tiene tarjeta de crédito" vale: 1, 0, 1, 1, 0, 1, 1, 1.
a) ¿Cuál es la proporción de clientes con tarjeta?
b) Escríbela como porcentaje.

**Ejercicio 4 (de razonamiento).** Dos grupos tienen la misma media de 50. El grupo A tiene desviación estándar 2 y el grupo B tiene desviación estándar 20. Describe en una frase cómo se ven los datos de cada grupo.